In [4]:
# If there's a conflict run this command first

# !pip uninstall -y chromadb

In [1]:
%pip install chromadb langchain-core langchain-community langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import posthog, deepeval, chromadb
print("posthog", posthog.__version__)
print("deepeval", deepeval.__version__)
print("chromadb", chromadb.__version__)

posthog 5.4.0
deepeval 3.6.9
chromadb 1.3.4


In [4]:
import json, random, math, csv, os
from collections import defaultdict
from statistics import quantiles
from deepeval.synthesizer import Synthesizer
from deepeval.synthesizer.config import FiltrationConfig, StylingConfig

random.seed(42)

SNAPSHOT = "test_apartments.json"
OUT_CSV = "golden_test_rag_style.csv"

def fmt_listing_line(r):
    addr = r.get("address","").strip()
    nbh  = r.get("neighborhood","").strip()
    price= r.get("price")
    beds = r.get("beds"); baths=r.get("baths")
    ams  = r.get("amenities") or []
    parts = []
    if addr or nbh: parts.append(f"{addr} — {nbh}".strip(" —"))
    meta = []
    if beds is not None:  meta.append(f"{beds} BR")
    if baths is not None: meta.append(f"{baths} BA")
    if price is not None: meta.append(f"${price:,}")
    if meta: parts.append(" / ".join(meta))
    if ams:  parts.append("Amenities: " + ", ".join(map(str, ams[:6])))
    return " | ".join(parts)

def build_context_block(listings):
    # mimic your “Top K listings retrieved for this turn” text block
    lines = []
    for i, r in enumerate(listings, 1):
        line = fmt_listing_line(r)
        if not line: 
            continue
        lines.append(f"{i}. {line}")
    return "\n".join(lines)

def to_price_band(p, qs):
    if p is None: return "unknown"
    q1, q2, q3 = qs
    if p <= q1: return "low"
    if p <= q2: return "mid"
    if p <= q3: return "upper"
    return "high"


In [5]:
# load data and stratify
with open(SNAPSHOT) as f:
    data = json.load(f)

prices = sorted([r["price"] for r in data if isinstance(r.get("price"), (int, float))])
qs = quantiles(prices, n=4) if len(prices) >= 4 else [min(prices or [0])]*3

buckets = defaultdict(list)
for r in data:
    ctx = fmt_listing_line(r)
    if not ctx: 
        continue
    nbh = (r.get("neighborhood") or "unknown").lower()
    pb  = to_price_band(r.get("price"), qs)
    buckets[(nbh, pb)].append(r)

In [7]:
# sample top-k blocks per stratum
TOP_K = 5
TARGET_GOLDENS = 300
MAX_PER_BUCKET = max(2, TARGET_GOLDENS // max(1,len(buckets)))

candidate_blocks = []
for key, recs in buckets.items():
    random.shuffle(recs)
    # take multiple disjoint blocks of size TOP_K
    for i in range(0, min(len(recs), MAX_PER_BUCKET*TOP_K), TOP_K):
        block = recs[i:i+TOP_K]
        if len(block) == TOP_K:
            candidate_blocks.append({
                "listings": block,
                "tags": [f"nbh:{key[0]}", f"price:{key[1]}"]
            })

random.shuffle(candidate_blocks)
candidate_blocks = candidate_blocks[:TARGET_GOLDENS]
len(candidate_blocks)

290

In [8]:
from deepeval.synthesizer import Synthesizer

def listing_str(rec):
    parts = []
    addr = rec.get("address","").strip()
    nbh  = rec.get("neighborhood","").strip()
    if addr or nbh: parts.append(f"{addr} — {nbh}".strip(" —"))
    meta = []
    if rec.get("beds") is not None:  meta.append(f"{rec['beds']} BR")
    if rec.get("baths") is not None: meta.append(f"{rec['baths']} BA")
    if rec.get("price") is not None: meta.append(f"${rec['price']:,}")
    if meta: parts.append(" / ".join(meta))
    return " | ".join(parts)

contexts = []      # e.g., [[ "1st listing line", "2nd listing line", ...], ...]
source_files = []  # e.g., ["id1,id2,id3,id4,id5", ...]

for topk in candidate_blocks:             
    context_lines = [listing_str(r) for r in topk if listing_str(r)]
    if len(context_lines) >= 2:
        contexts.append(context_lines)     # one context = many short strings
        source_files.append(",".join(str(r.get("id","")) for r in topk))

assert len(contexts) == len(source_files)

synth = Synthesizer()
synth.generate_goldens_from_contexts(
    contexts=contexts,                     # required
    include_expected_output=True,          # optional (defaults True)
    max_goldens_per_context=1,             # optional
    source_files=source_files              # optional; must match len(contexts)
)

AttributeError: 'str' object has no attribute 'get'

In [ ]:
synth = Synthesizer()
synth.generate_goldens_from_docs(
    document_paths=["docs/a.txt","docs/b.md","docs/c.pdf"],   # required
    include_expected_output=True,                             # optional (default True)
    max_goldens_per_context=2,                                # optional (default 2)
    context_construction_config=ContextConstructionConfig(    # optional
        max_contexts_per_document=2,
        max_context_length=3,
        chunk_size=800,
        chunk_overlap=0
    )
)